[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/review_1_python/blob/main/sesiones/refuerzo_python_puro.ipynb)

# Refuerzo · Diccionarios, estado acumulativo y `key`

**Python puro** · ⏱️ Duración estimada: 3 h 40 min

## 🎯 Objetivos
Al terminar podrás:
1. Agrupar datos en un diccionario (conteos, totales, promedios) con `.get()` y sin `KeyError`.
2. Llevar un estado (saldo, stock) que arranca en su valor inicial y detectar eventos mientras avanza.
3. Buscar extremos a mano con `None` como centinela, guardando el registro completo y decidiendo los empates.
4. Usar `min`, `max` y `sorted` con `key`, y combinar `zip`, `enumerate` y `dict(zip(...))`.
5. Resolver un extracto bancario completo, primero a tu manera y luego en un solo recorrido.

## 📋 Qué debes saber antes
Listas, tuplas y diccionarios básicos, `if`/`elif`/`else`, `for`, `while`, `break`, `continue` y f-strings.

## 🧭 Cómo trabajar este notebook
- Guarda una copia en tu Drive (**Archivo → Guardar una copia en Drive**).
- Ejecuta el **⚙️ Setup** al empezar y las celdas **en orden**, con **Shift + Enter**.
- En cada ejemplo, lee la línea 🔮 y escribe tu predicción en el comentario **antes** de ejecutar.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí` y luego ejecuta ✅ **Verificar**. Si aparece ❌, lee el motivo: te dice qué error típico cometiste.
- Si te atascas, abre la 💡 **Pista** (hay dos, de menos a más ayuda). Las 🔒 **soluciones** están en el anexo final.
- Si modificas los datos por error, vuelve a ejecutar el setup.

| Bloque | Tiempo |
|---|---|
| Parte 1 · Diccionarios: agrupar y acumular (secciones 1-4) | 50 min |
| Parte 2 · Estado acumulativo y control de flujo (secciones 5-8) | 70 min |
| Parte 3 · Tuplas, `key`, `zip` y tablas (secciones 9-12) | 50 min |
| 🏋️ Reto final + 🚀 Nivel pro | 40 min |
| Portada y cierre | 10 min |

## 🎯 Errores típicos que vas a cazar hoy
1. Arrancar el estado en 0 y «corregirlo» al final: el resultado final coincide, pero los valores intermedios y los eventos quedan mal.
2. Filtrar el monto 0 donde no hace falta (en un saldo) y no filtrarlo donde sí importa (en un diccionario, donde crea una clave).
3. Usar 0 como valor inicial de un extremo: funciona por accidente con egresos y falla cuando todo es positivo.
4. Leer una clave que aún no existe (`KeyError`) o usar `.get()` sin valor por defecto (`None + 1`).
5. Escribir un `for` por cada resultado en vez de llevar varios acumuladores en un solo recorrido.
6. Entregar `477.5799999999999` en lugar de redondear, o comparar decimales con `==`.
7. `min(lista_de_tuplas)` sin `key`, que compara por la fecha y no por el monto.
8. Un `continue` que se salta un caso válido antes del chequeo importante.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Carga los datos de práctica y las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de práctica y carga las funciones que revisan tus respuestas.
import copy
import math

# ======================================================================
# Datos de práctica
# ======================================================================

# ---------- Parte 1 · Diccionarios ----------
ventas_bodega = [  # (fecha, categoria, monto en soles)
    ("2026-05-11", "abarrotes", 24.90),
    ("2026-05-11", "lácteos", 12.80),
    ("2026-05-11", "snacks", 2.20),
    ("2026-05-12", "bebidas", 7.50),
    ("2026-05-12", "abarrotes", 18.35),
    ("2026-05-12", "lácteos", 9.60),
    ("2026-05-13", "limpieza", 14.20),
    ("2026-05-13", "abarrotes", 7.10),
    ("2026-05-13", "snacks", 1.10),
    ("2026-05-14", "lácteos", 15.45),
    ("2026-05-14", "bebidas", 5.00),
    ("2026-05-14", "snacks", 3.50),
    ("2026-05-14", "limpieza", 8.90),
    ("2026-05-15", "abarrotes", 31.60),
    ("2026-05-15", "bebidas", 6.30),
]

movimientos_junio = [  # (fecha, concepto, monto): + ingreso, - egreso, 0 ajuste
    ("2026-06-01", "sueldo", 1500.00),
    ("2026-06-02", "alquiler", -600.00),
    ("2026-06-03", "mercado", -85.40),
    ("2026-06-04", "ajuste", 0.0),
    ("2026-06-06", "yape", 120.00),
    ("2026-06-07", "mercado", -62.30),
    ("2026-06-09", "transporte", -18.50),
    ("2026-06-12", "mercado", -47.15),
    ("2026-06-14", "ajuste", 0.0),
    ("2026-06-15", "transporte", -21.00),
]

compras_mes = [  # (fecha, categoria, monto) compras a proveedores
    ("2026-07-01", "abarrotes", 242.50),
    ("2026-07-01", "limpieza", 58.90),
    ("2026-07-03", "bebidas", 365.40),
    ("2026-07-04", "abarrotes", 215.75),
    ("2026-07-06", "lácteos", 89.30),
    ("2026-07-08", "limpieza", 44.10),
    ("2026-07-09", "bebidas", 98.60),
    ("2026-07-11", "abarrotes", 187.20),
    ("2026-07-12", "lácteos", 76.90),
    ("2026-07-14", "snacks", 63.15),
    ("2026-07-15", "abarrotes", 110.00),
    ("2026-07-15", "limpieza", 39.95),
]

# ---------- Parte 2 · Estado acumulativo ----------
saldo_inicial_caja = 250.0
movimientos_caja = [
    ("2026-06-01", "proveedor", -180.50),
    ("2026-06-02", "ventas del día", 95.20),
    ("2026-06-03", "ajuste", 0.0),
    ("2026-06-04", "luz", -64.30),
    ("2026-06-05", "ventas del día", 132.75),
    ("2026-06-06", "agua", -28.10),
    ("2026-06-07", "proveedor", -90.00),
    ("2026-06-08", "ventas del día", 210.40),
]

saldo_inicial_cuenta = 150.0
movimientos_cuenta = [
    ("2026-07-01", "proveedor", -90.00),
    ("2026-07-02", "ventas", 45.50),
    ("2026-07-03", "alquiler", -180.00),
    ("2026-07-04", "ajuste", 0.00),
    ("2026-07-05", "ventas", 120.00),
    ("2026-07-06", "luz", -62.30),
    ("2026-07-07", "ventas", 38.40),
    ("2026-07-08", "agua", -15.20),
]

saldo_inicial_ahorro = 500.0
movimientos_ahorro = [
    ("2026-07-01", "proveedor", -120.00),
    ("2026-07-02", "ventas", 85.50),
    ("2026-07-03", "agua", -31.20),
    ("2026-07-04", "luz", -58.90),
    ("2026-07-05", "ventas", 60.00),
    ("2026-07-06", "proveedor", -140.25),
]

saldo_inicial_agosto = 400.0
movimientos_agosto = [
    ("2026-08-01", "proveedor", -150.00),
    ("2026-08-02", "ventas", 80.00),
    ("2026-08-03", "luz", -95.50),
    ("2026-08-04", "ventas", 120.30),
    ("2026-08-05", "alquiler", -200.00),
    ("2026-08-06", "ajuste", 0.00),
    ("2026-08-07", "ventas", 99.00),
    ("2026-08-08", "agua", -44.10),
]

stock_inicial = 40      # unidades de aceite de 1 L
stock_minimo = 15
kardex = [  # (fecha, tipo, cantidad)
    ("2026-09-01", "salida", 12),
    ("2026-09-02", "salida", 9),
    ("2026-09-03", "salida", 7),
    ("2026-09-04", "salida", 4),
    ("2026-09-05", "entrada", 20),
    ("2026-09-06", "salida", 15),
    ("2026-09-07", "salida", 3),
    ("2026-09-08", "entrada", 30),
    ("2026-09-09", "salida", 15),
    ("2026-09-10", "salida", 15),
]

# ---------- Parte 3 · Tuplas y key ----------
lista_prueba = [
    ("2026-05-01", 30.0), ("2026-04-20", 90.0), ("2026-05-01", 25.0),
    ("2026-04-27", 15.5), ("2026-05-09", 12.0), ("2026-04-20", 40.0),
]

movimientos_abril = [
    ("2026-04-01", "mercado", -45.20),
    ("2026-04-03", "yape", 230.00),
    ("2026-04-06", "sueldo", 1800.00),
    ("2026-04-08", "alquiler", -650.00),
    ("2026-04-10", "transporte", -58.00),
    ("2026-04-13", "luz", -96.40),
    ("2026-04-17", "yape", 415.50),
    ("2026-04-19", "internet", -79.90),
    ("2026-04-22", "ajuste", 0.00),
    ("2026-04-25", "mercado", -188.75),
]

fechas_semana = ["2026-10-05", "2026-10-06", "2026-10-07", "2026-10-08", "2026-10-09", "2026-10-10", "2026-10-11"]
ventas_dia = [412.50, 388.00, 455.20, 455.20, 301.75, 520.60, 520.60]
clientes_dia = [96, 90, 101, 88, 70, 115, 104]

locales = ["Centro", "Mercado", "Paradero"]
meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio"]
ventas_mensuales = [  # filas: locales; columnas: meses
    [4200, 3980, 4510, 4510, 4105, 3890],
    [5120, 4870, 5340, 5010, 5490, 5230],
    [2890, 3150, 2760, 3320, 3040, 3320],
]

# ---------- Reto final y nivel pro ----------
saldo_inicial = 300.0
movimientos = [
    ("2026-10-01", "ventas", 180.50),
    ("2026-10-02", "proveedor", -220.30),
    ("2026-10-03", "luz", -85.60),
    ("2026-10-04", "proveedor", -350.00),
    ("2026-10-05", "ajuste", 0.00),
    ("2026-10-06", "ventas", 240.00),
    ("2026-10-07", "transporte", -32.50),
    ("2026-10-08", "alquiler", -350.00),
    ("2026-10-09", "yape", 95.75),
    ("2026-10-10", "ventas", 210.40),
    ("2026-10-11", "ventas", 150.25),
    ("2026-10-12", "internet", -69.90),
    ("2026-10-13", "proveedor", -60.20),
    ("2026-10-14", "yape", 120.00),
    ("2026-10-15", "agua", -28.60),
    ("2026-10-16", "proveedor", -41.15),
]

# Copia privada: los verificadores no dependen de lo que cambies arriba.
_D = copy.deepcopy({k: globals()[k] for k in [
    "ventas_bodega", "movimientos_junio", "compras_mes",
    "saldo_inicial_caja", "movimientos_caja", "saldo_inicial_cuenta", "movimientos_cuenta",
    "saldo_inicial_ahorro", "movimientos_ahorro", "saldo_inicial_agosto", "movimientos_agosto",
    "stock_inicial", "stock_minimo", "kardex",
    "lista_prueba", "movimientos_abril", "fechas_semana", "ventas_dia", "clientes_dia",
    "locales", "meses", "ventas_mensuales", "saldo_inicial", "movimientos",
]})

# ======================================================================
# Herramientas de verificación
# ======================================================================
_FALTA = object()


def _igual(a, b, tol=0.0051):
    """Compara exigiendo el mismo tipo en None/bool/str y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return isinstance(a, dict) and set(a) == set(b) and all(_igual(a[k], b[k], tol) for k in b)
    return type(a) is type(b) and a == b


def _redondeado(valor):
    """True si ningún decimal de `valor` tiene más de 2 cifras."""
    if isinstance(valor, float):
        return round(valor, 2) == valor
    if isinstance(valor, dict):
        return all(_redondeado(v) for v in valor.values())
    if isinstance(valor, (list, tuple)):
        return all(_redondeado(v) for v in valor)
    return True


def _corto(valor, n=70):
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _saldos(inicial, movs):
    """Saldo después de cada movimiento."""
    resultado = []
    for i in range(len(movs)):
        resultado.append(inicial + math.fsum(m[2] for m in movs[:i + 1]))
    return resultado


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and type(valor) is not tipo:
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {tipo.__name__}.")
            return _FALTA
        return valor

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", trampas=(), reglas=(),
              redondeo=False):
        """Compara sin mostrar el valor esperado. `trampas`: (valor_de_un_error_típico, mensaje).
        `reglas`: (función que detecta un error típico, mensaje)."""
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        if _igual(v, esperado):
            if redondeo and not _redondeado(v):
                self.mal(f"`{nombre}` tiene el valor correcto, pero sin redondear: usa round(..., 2) al entregar el resultado.")
            else:
                self.ok(f"`{nombre}` es correcto.")
            return
        for regla, mensaje in reglas:
            try:
                activa = regla(v)
            except Exception:
                activa = False
            if activa:
                self.mal(f"`{nombre}`: {mensaje}")
                return
        for trampa, mensaje in trampas:
            if _igual(v, trampa):
                self.mal(f"`{nombre}` vale {_corto(v)}: {mensaje}")
                return
        self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def prediccion(self, nombre, esperado):
        v = self.var(nombre)
        if v is _FALTA:
            return
        normal = v.strip().lower() if isinstance(v, str) else v
        objetivo = esperado.strip().lower() if isinstance(esperado, str) else esperado
        if _igual(normal, objetivo, tol=1e-9):
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _sin_cambios(r, nombre):
    if globals().get(nombre) != _D[nombre]:
        r.mal(f"`{nombre}` cambió. No debes modificar los datos originales: vuelve a ejecutar el setup y revisa tu código.")


def _por_clave(registros, i_clave, valor):
    """Diccionario clave -> lista de valores, en orden de aparición."""
    grupos = {}
    for reg in registros:
        grupos.setdefault(reg[i_clave], []).append(valor(reg))
    return grupos


# ---------------------------- Parte 1 ----------------------------
def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    grupos = _por_clave(_D["ventas_bodega"], 1, lambda v: v[2])
    esperado = {c: len(ms) for c, ms in grupos.items()}
    r.valor("conteo_por_categoria", esperado, dict, "cada venta suma 1 a su categoría",
            reglas=[(lambda d: any(type(x) is not int for x in d.values()),
                     "hay valores con decimales: eso es un total en soles, no un conteo. Suma 1 por venta, no el monto.")],
            trampas=[({c: n - 1 for c, n in esperado.items()},
                      "a cada categoría le falta 1. ¿Inicializaste en 0 dentro del `else`? La primera venta también cuenta.")])
    _sin_cambios(r, "ventas_bodega")
    r.fin()

    r = _Revision("Ejercicio 1 · Parte B")
    r.prediccion("pred_in_cero", "azúcar" in {"azúcar": 0})
    r.prediccion("pred_in_valor", 12 in {"arroz": 12})
    try:
        {"arroz": 12}["Arroz"]
        error = "ninguno"
    except Exception as e:
        error = type(e).__name__
    r.prediccion("pred_error", error)
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    grupos = _por_clave(_D["ventas_bodega"], 1, lambda v: v[2])
    esperado = {c: round(math.fsum(ms), 2) for c, ms in grupos.items()}
    sin_primero = {c: round(math.fsum(ms[1:]), 2) for c, ms in grupos.items()}
    r.valor("total_por_categoria", esperado, dict, "suma todos los montos de cada categoría",
            trampas=[(sin_primero, "a cada categoría le falta su primer monto: ¿la inicializaste en 0 en vez de acumular desde la primera venta?")],
            redondeo=True)
    _sin_cambios(r, "ventas_bodega")
    r.fin()

    r = _Revision("Ejercicio 2 · Parte B")
    r.prediccion("pred_igual", 0.1 + 0.2 == 0.3)
    r.prediccion("pred_redondeado", round(0.1 + 0.2, 2) == 0.3)
    try:
        {"lunes": 120.0}.get("martes") + 50
        error = "ninguno"
    except Exception as e:
        error = type(e).__name__
    r.prediccion("pred_error_get", error)
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3")
    movs = _D["movimientos_junio"]
    grupos = _por_clave([m for m in movs if m[2] < 0], 1, lambda m: -m[2])
    esperado = {c: round(math.fsum(ms), 2) for c, ms in grupos.items()}
    ingresos = {m[1] for m in movs if m[2] > 0}
    ceros = {m[1] for m in movs if m[2] == 0}
    r.valor("egresos_por_concepto", esperado, dict, "suma, en positivo, los egresos de cada concepto",
            reglas=[(lambda g: bool(ingresos & set(g)),
                     "tiene conceptos de ingresos: el filtro `monto != 0` deja pasar los ingresos. Solo cuentan los montos negativos."),
                    (lambda g: bool(ceros & set(g)),
                     "tiene un concepto que solo tuvo montos 0: aquí sí hay que filtrarlo, o crea una clave que no debería existir."),
                    (lambda g: any(x < 0 for x in g.values()),
                     "hay gastos negativos: guárdalos en positivo (suma `-monto`).")],
            redondeo=True)
    _sin_cambios(r, "movimientos_junio")
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4")
    grupos = _por_clave(_D["compras_mes"], 1, lambda m: m[2])
    totales = {c: math.fsum(ms) for c, ms in grupos.items()}
    r.valor("compras_total", {c: round(t, 2) for c, t in totales.items()}, dict,
            "suma los montos de cada categoría", redondeo=True,
            trampas=[({c: round(math.fsum(ms[1:]), 2) for c, ms in grupos.items()},
                      "a cada categoría le falta su primera compra.")])
    r.valor("compras_cantidad", {c: len(ms) for c, ms in grupos.items()}, dict,
            "cuenta 1 por compra (no sumes el monto)")
    r.valor("compras_promedio", {c: round(totales[c] / len(ms), 2) for c, ms in grupos.items()}, dict,
            "promedio = total / cantidad de la categoría", redondeo=True)
    mayor_suelta = sorted(_D["compras_mes"], key=lambda m: -m[2])[0][1]
    r.valor("categoria_mas_compras", sorted(totales, key=lambda c: -totales[c])[0], str,
            "debería ser la categoría con mayor TOTAL",
            trampas=[(mayor_suelta, "es la categoría de la compra suelta más grande; compara los TOTALES por categoría."),
                     (sorted(totales, key=lambda c: totales[c])[0], "es la categoría con MENOR total: revisa la comparación.")])
    _sin_cambios(r, "compras_mes")
    r.fin()


# ---------------------------- Parte 2 ----------------------------
def check_ejercicio_5():
    r = _Revision("Ejercicio 5")
    movs = _D["movimientos_caja"]
    saldos = _saldos(_D["saldo_inicial_caja"], movs)
    r.valor("saldos_caja", [round(s, 2) for s in saldos], list, "cada saldo es el anterior más el monto del día",
            reglas=[(lambda v: len(v) != len(movs),
                     f"debe tener un saldo por movimiento ({len(movs)}). ¿Saltaste el monto 0? Sumar 0 no cambia el saldo, pero ese día existe.")],
            trampas=[([round(s, 2) for s in _saldos(0, movs)],
                      "¿arrancaste el saldo en 0? Debe arrancar en `saldo_inicial_caja`.")],
            redondeo=True)
    r.valor("saldo_cierre", round(saldos[-1], 2), float, "debería ser el último saldo",
            trampas=[(round(saldos[-1] - _D["saldo_inicial_caja"], 2), "falta el saldo inicial.")], redondeo=True)
    _sin_cambios(r, "movimientos_caja")
    r.fin()


def check_ejercicio_6():
    r = _Revision("Ejercicio 6")
    movs = _D["movimientos_cuenta"]
    saldos = _saldos(_D["saldo_inicial_cuenta"], movs)
    rojos = [m[0] for m, s in zip(movs, saldos) if s < 0]
    anteriores = [_D["saldo_inicial_cuenta"]] + saldos[:-1]
    desde_cero = [m[0] for m, s in zip(movs, _saldos(0, movs)) if s < 0]
    r.valor("primer_rojo", rojos[0], str, "debería ser la fecha del primer saldo menor que 0",
            trampas=[(rojos[-1], "es el ÚLTIMO día en rojo: ¿te falta el `break`?"),
                     (desde_cero[0], "¿arrancaste el saldo en 0 en vez de en `saldo_inicial_cuenta`?")])
    r.valor("dias_en_rojo", rojos, list, "deberían ser todas las fechas con saldo menor que 0, en orden",
            trampas=[([m[0] for m, s in zip(movs, saldos) if s < 0 and m[2] != 0],
                      "falta un día: ¿un `continue` con el monto 0 se saltó el chequeo? Ese día el saldo seguía en rojo.")])
    r.valor("entradas_en_rojo", sum(1 for a, s in zip(anteriores, saldos) if a >= 0 > s), int,
            "cuenta las veces que el saldo PASA de >= 0 a < 0",
            trampas=[(len(rojos), "eso es la cantidad de días en rojo, no de entradas: compara con el saldo anterior.")])
    ahorro = [m[0] for m, s in zip(_D["movimientos_ahorro"], _saldos(_D["saldo_inicial_ahorro"], _D["movimientos_ahorro"])) if s < 0]
    r.valor("primer_rojo_ahorro", ahorro[0] if ahorro else None, None,
            "esta cuenta nunca queda en rojo: debe quedar en `None` (no en 0, \"\" ni una fecha)")
    _sin_cambios(r, "movimientos_cuenta")
    _sin_cambios(r, "movimientos_ahorro")
    r.fin()


def check_ejercicio_7():
    r = _Revision("Ejercicio 7 · Parte A")
    movs = _D["movimientos_agosto"]
    saldos = _saldos(_D["saldo_inicial_agosto"], movs)
    minimo = sorted(saldos)[0]
    fechas_min = [m[0] for m, s in zip(movs, saldos) if s == minimo]
    r.valor("saldo_minimo", round(minimo, 2), None, "debería ser el menor saldo tras un movimiento",
            trampas=[(0, "todos los saldos son positivos, así que el 0 inicial «gana» y nunca se reemplaza. Usa `None` como centinela.")],
            redondeo=True)
    r.valor("fecha_saldo_minimo", fechas_min[0], None, "debería ser la fecha del saldo mínimo",
            trampas=[(None, "nunca se actualizó: con el mínimo inicializado en 0 ningún saldo lo supera. Usa `None` como centinela."),
                     (fechas_min[-1], "hay un empate y te quedaste con el último: usa `<` (no `<=`) para conservar el primero.")])
    _sin_cambios(r, "movimientos_agosto")
    r.fin()

    r = _Revision("Ejercicio 7 · Parte B")
    precios = [("Proveedor A", 12.5), ("Proveedor B", 11.9), ("Proveedor C", 11.9), ("Proveedor D", 13.0)]
    for nombre, estricto in (("pred_con_menor", True), ("pred_con_menor_igual", False)):
        mejor = None
        for p in precios:
            if mejor is None or (p[1] < mejor[1] if estricto else p[1] <= mejor[1]):
                mejor = p
        r.prediccion(nombre, mejor[0])
    r.fin()


def check_ejercicio_8():
    r = _Revision("Ejercicio 8")
    minimo = _D["stock_minimo"]
    stocks = [_D["stock_inicial"]]
    for _, tipo, cant in _D["kardex"]:
        stocks.append(stocks[-1] + (cant if tipo == "entrada" else -cant))
    despues = stocks[1:]
    fechas = [m[0] for m in _D["kardex"]]
    r.valor("stock_final", stocks[-1], int, "las entradas suman y las salidas restan, desde `stock_inicial`")
    r.valor("fechas_bajo_minimo", [f for f, s in zip(fechas, despues) if s < minimo], list,
            "compara el stock DESPUÉS de cada movimiento con `stock_minimo`")
    recuperacion = [f for f, a, s in zip(fechas, stocks, despues) if a < minimo < s]
    r.valor("fecha_recuperacion", recuperacion[0] if recuperacion else None, None,
            "es la primera vez que el stock pasa de estar bajo el mínimo a superarlo",
            trampas=[([f for f, s in zip(fechas, despues) if s > minimo][0],
                      "ese día el stock supera el mínimo, pero no venía de estar bajo: necesitas el stock anterior.")])
    salidas = [m for m in _D["kardex"] if m[1] == "salida"]
    tope = sorted(m[2] for m in salidas)[-1]
    empatadas = [m for m in salidas if m[2] == tope]
    r.valor("mayor_salida", empatadas[0], tuple, "debería ser la tupla completa de la mayor salida",
            trampas=[(sorted(_D["kardex"], key=lambda m: m[2])[-1], "es una ENTRADA: busca solo entre las salidas."),
                     (empatadas[-1], "hay un empate y te quedaste con la última: usa `>` (no `>=`).")])
    _sin_cambios(r, "kardex")
    r.fin()


# ---------------------------- Parte 3 ----------------------------
def check_ejercicio_9():
    r = _Revision("Ejercicio 9")
    r.prediccion("pred_a", ("2026-11-03", 10) < ("2026-11-03", 9))
    r.prediccion("pred_b", ("b", 1) < ("a", 99))
    r.prediccion("pred_c", (1, 500) < (1, 1000))
    r.prediccion("pred_d", ("2026-10-5", 0) < ("2026-10-15", 0))
    r.prediccion("pred_e", "100" < "25")
    r.prediccion("pred_f", (7, 3) < (7, 3, 0))
    r.prediccion("pred_min", min(_D["lista_prueba"]))
    r.fin()


def check_ejercicio_10():
    r = _Revision("Ejercicio 10")
    movs = _D["movimientos_abril"]
    por_monto = sorted(movs, key=lambda m: m[2])
    egresos = [m for m in por_monto if m[2] < 0]
    r.valor("mayor_ingreso", por_monto[-1], tuple, "debería ser el movimiento con el monto más alto",
            trampas=[(max(movs), "es el movimiento más RECIENTE: `max` sin `key` compara por la fecha.")])
    r.valor("mayor_egreso_abril", por_monto[0], tuple, "debería ser el movimiento con el monto más bajo",
            trampas=[(min(movs), "es el movimiento más ANTIGUO: `min` sin `key` compara por la fecha.")])
    solo_egresos = [m for m in movs if m[2] < 0]
    r.valor("top3_egresos", egresos[:3], list, "deberían ser los 3 egresos más grandes, del más negativo al menos negativo",
            trampas=[(sorted(solo_egresos, reverse=True)[:3], "están ordenados por FECHA: `sorted` sin `key` compara la tupla completa."),
                     (sorted(solo_egresos, key=lambda m: m[2], reverse=True)[:3],
                      "con `reverse=True` los negativos quedan al revés: primero el egreso más pequeño.")])
    _sin_cambios(r, "movimientos_abril")
    r.fin()


def check_ejercicio_11():
    r = _Revision("Ejercicio 11")
    f, v, c = _D["fechas_semana"], _D["ventas_dia"], _D["clientes_dia"]
    registros = [(i + 1, f[i], v[i], c[i]) for i in range(len(f))]
    ranking = sorted(registros, key=lambda t: (-t[2], t[3]))
    r.valor("venta_por_fecha", {f[i]: v[i] for i in range(len(f))}, dict, "cada fecha debería apuntar a su venta")
    r.valor("ranking", ranking, list, "ordena por venta de mayor a menor y, si empatan, menos clientes primero",
            reglas=[(lambda x: sorted(t[0] for t in x) == list(range(len(f))),
                     "los números de día empiezan en 0: usa `enumerate(..., start=1)`.")],
            trampas=[(sorted(registros, key=lambda t: -t[2]), "falta el desempate: si la venta empata, va primero el de menos clientes."),
                     (sorted(registros, key=lambda t: (-t[2], -t[3])), "el desempate está invertido: menos clientes va primero.")])
    r.valor("dia_ganador", ranking[0][0], int, "debería ser el número de día (1 a 7) del primer lugar",
            trampas=[(ranking[1][0], "¿aplicaste el desempate por clientes? ¿empezaste a contar en 1?")])
    r.valor("fecha_ganadora", ranking[0][1], str, "debería ser la fecha del primer lugar del ranking")
    for nombre in ("fechas_semana", "ventas_dia", "clientes_dia"):
        _sin_cambios(r, nombre)
    r.fin()


def check_ejercicio_12():
    r = _Revision("Ejercicio 12")
    tabla = _D["ventas_mensuales"]
    totales = {loc: sum(fila) for loc, fila in zip(_D["locales"], tabla)}
    primero, ultimo, base_cero = {}, {}, {}
    for loc, fila in zip(_D["locales"], tabla):
        posiciones = [j for j in range(len(fila)) if fila[j] == sorted(fila)[-1]]
        primero[loc] = posiciones[0] + 1
        ultimo[loc] = posiciones[-1] + 1
        base_cero[loc] = posiciones[0]
    r.valor("total_por_local", totales, dict, "cada local debería apuntar a la suma de su fila")
    r.valor("mejor_mes_por_local", primero, dict, "cada local debería apuntar al número de mes (1 a 6) de su mayor venta",
            trampas=[(ultimo, "en los empates te quedaste con el último mes: usa `>` (no `>=`), o `.index()`, que da el primero."),
                     (base_cero, "los meses van del 1 al 6: a la posición súmale 1.")])
    r.valor("local_top", sorted(totales, key=lambda k: -totales[k])[0], str, "debería ser el local con mayor venta total")
    _sin_cambios(r, "ventas_mensuales")
    r.fin()


# ------------------------- Reto y nivel pro -------------------------
def _reto_esperado():
    si, movs = _D["saldo_inicial"], _D["movimientos"]
    saldos = _saldos(si, movs)
    anteriores = [si] + saldos[:-1]
    grupos = _por_clave([m for m in movs if m[2] < 0], 1, lambda m: -m[2])
    gasto = {c: round(math.fsum(ms), 2) for c, ms in grupos.items()}
    return movs, saldos, anteriores, gasto


def check_reto(parte="A"):
    r = _Revision(f"Reto final · Parte {parte}")
    si = _D["saldo_inicial"]
    movs, saldos, anteriores, gasto = _reto_esperado()
    montos = [m[2] for m in movs]
    r.valor("saldo_final", round(saldos[-1], 2), float, "debería ser el saldo inicial más todos los montos",
            trampas=[(round(math.fsum(montos), 2), "falta el saldo inicial: el saldo arranca en `saldo_inicial`, no en 0.")],
            redondeo=True)
    r.valor("n_ingresos", sum(1 for x in montos if x > 0), int, "cuenta solo los montos mayores que 0",
            trampas=[(sum(1 for x in montos if x >= 0), "¿contaste el monto 0 como ingreso? El ajuste no es ingreso ni egreso.")])
    r.valor("n_egresos", sum(1 for x in montos if x < 0), int, "cuenta solo los montos menores que 0",
            trampas=[(sum(1 for x in montos if x <= 0), "¿contaste el monto 0 como egreso? El ajuste no es ingreso ni egreso.")])
    ingresos_o_cero = {m[1] for m in movs if m[2] >= 0} - set(gasto)
    r.valor("gasto_por_concepto", gasto, dict, "algún total no coincide: suma, en positivo, todos los egresos de cada concepto",
            reglas=[(lambda g: bool(ingresos_o_cero & set(g)),
                     "tiene conceptos que no son egresos (ingresos o el ajuste de 0): solo cuentan los montos negativos."),
                    (lambda g: any(x < 0 for x in g.values()), "hay gastos negativos: guárdalos en positivo."),
                    (lambda g: set(g) != set(gasto), "faltan conceptos: ¿incluiste los que tienen un solo egreso?")],
            redondeo=True)
    por_monto = sorted(movs, key=lambda m: m[2])
    empatados = [m for m in movs if m[2] == por_monto[0][2]]
    r.valor("mayor_egreso", por_monto[0], tuple, "debería ser la tupla completa del movimiento con el monto más negativo",
            trampas=[(min(movs), "es el movimiento más ANTIGUO: `min(movimientos)` sin `key` compara por la fecha."),
                     (empatados[-1], "hay un empate y te quedaste con el último: usa `<` (no `<=`) para conservar el primero.")])
    rojos = [m[0] for m, s in zip(movs, saldos) if s < 0]
    r.valor("dias_sobregiro", rojos, list, "deberían ser las fechas en que el saldo, tras el movimiento, quedó por debajo de 0",
            trampas=[([m[0] for m, s in zip(movs, saldos) if s < 0 and m[2] != 0],
                      "falta un día: ¿un `continue` con el monto 0 se saltó el chequeo? Ese día la cuenta seguía en sobregiro."),
                     ([m[0] for m, s in zip(movs, _saldos(0, movs)) if s < 0],
                      "¿arrancaste el saldo en 0? Debe arrancar en `saldo_inicial`.")])
    r.valor("veces_en_sobregiro", sum(1 for a, s in zip(anteriores, saldos) if a >= 0 > s), int,
            "cuenta las veces que el saldo PASA de >= 0 a < 0",
            trampas=[(len(rojos), "eso es la cantidad de días en sobregiro, no de entradas: compara con el saldo anterior.")])
    r.valor("primer_dia_sobregiro", rojos[0] if rojos else None, None, "debería ser la primera fecha con saldo menor que 0",
            trampas=[(rojos[-1] if rojos else None, "es el ÚLTIMO día en sobregiro: ¿te falta el `break` o la condición `is None`?")])
    mayor_suelto = por_monto[0][1]
    r.valor("concepto_mas_gasto", sorted(gasto, key=lambda c: -gasto[c])[0], str,
            "debería ser el concepto con mayor gasto TOTAL",
            trampas=[] if mayor_suelto == sorted(gasto, key=lambda c: -gasto[c])[0] else
            [(mayor_suelto, "es el concepto del egreso suelto más grande; compara los TOTALES.")])
    _sin_cambios(r, "movimientos")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    movs, saldos, _, gasto = _reto_esperado()
    r.valor("saldo_por_fecha", {m[0]: round(s, 2) for m, s in zip(movs, saldos)}, dict,
            "cada fecha debería apuntar al saldo que quedó tras su movimiento", redondeo=True)
    ranking = sorted(gasto.items(), key=lambda kv: (-kv[1], kv[0]))
    r.valor("ranking_conceptos", ranking, list, "deberían ser tuplas (concepto, gasto) de mayor a menor gasto",
            trampas=[(ranking[::-1], "está de menor a mayor: invierte el orden (signo menos en la `key` o `reverse=True`).")])
    racha = mejor = 0
    for s in saldos:
        racha = racha + 1 if s < 0 else 0
        mejor = mejor if mejor >= racha else racha
    r.valor("racha_mas_larga", mejor, int, "cuenta la mayor cantidad de movimientos SEGUIDOS con saldo en rojo",
            trampas=[(sum(1 for s in saldos if s < 0), "eso es el total de días en rojo; la racha son días seguidos (se reinicia al salir del rojo).")])
    _sin_cambios(r, "movimientos")
    r.fin()


print("✅ Setup listo. Datos cargados y verificadores activos.")

### 📦 Tus datos de hoy
Todos los ejercicios usan estas variables. Son datos fijos: siempre salen iguales.

In [ ]:
def mostrar(nombre, valor):
    if isinstance(valor, list) and valor and isinstance(valor[0], (tuple, list)):
        print(f"{nombre} =")
        for elemento in valor:
            print("   ", elemento)
    else:
        print(f"{nombre} = {valor}")

print("🧺 Parte 1 · Diccionarios")
for nombre in ["ventas_bodega", "movimientos_junio", "compras_mes"]:
    mostrar(nombre, globals()[nombre])
print("\n💰 Parte 2 · Estado acumulativo")
for nombre in ["saldo_inicial_caja", "movimientos_caja", "saldo_inicial_cuenta", "movimientos_cuenta",
               "saldo_inicial_ahorro", "movimientos_ahorro", "saldo_inicial_agosto", "movimientos_agosto",
               "stock_inicial", "stock_minimo", "kardex"]:
    mostrar(nombre, globals()[nombre])
print("\n🔢 Parte 3 · Tuplas, key y tablas")
for nombre in ["lista_prueba", "movimientos_abril", "fechas_semana", "ventas_dia", "clientes_dia",
               "locales", "meses", "ventas_mensuales"]:
    mostrar(nombre, globals()[nombre])
print("\n🏋️ Reto final")
for nombre in ["saldo_inicial", "movimientos"]:
    mostrar(nombre, globals()[nombre])

---
# Parte 1 · Diccionarios: agrupar y acumular
⏱️ 50 minutos · Secciones 1 a 4

---
## 1. El ciclo de vida de una clave

### 📘 Concepto
Una clave **nace cuando le asignas un valor** (`d["leche"] = 5`). Hasta entonces no existe:

| Operación | Si la clave existe | Si no existe |
|---|---|---|
| `d[clave]` | devuelve el valor | `KeyError`: el programa se detiene |
| `clave in d` | `True` | `False` |
| `d[clave] = valor` | cambia el valor | **crea** la clave |

`in` pregunta por las **claves**, nunca por los valores.

**Patrón clásico para agrupar** (categoría → conteo o total):
```python
if categoria in conteo:
    conteo[categoria] += 1      # ya existía: acumular
else:
    conteo[categoria] = 1       # primera vez: nace con el valor de ESTA vuelta
```
⚠️ Si en el `else` pones `= 0`, la primera aparición de cada categoría se pierde.

In [ ]:
# 🔮 Predice antes de ejecutar: ¿con cuánto termina "lácteos"? ¿En qué vuelta nace "bebidas"?
# Mi predicción:

pedidos_ej = [("lácteos", 8.50), ("abarrotes", 23.00), ("lácteos", 6.50), ("bebidas", 4.00), ("lácteos", 9.00)]

conteo_ej = {}
print(f"{'vuelta':<7}| {'elemento':<22}| {'conteo':<44}| evento")
for vuelta, (categoria, monto) in enumerate(pedidos_ej, start=1):
    if categoria in conteo_ej:
        conteo_ej[categoria] += 1
        evento = "ya existía → acumula"
    else:
        conteo_ej[categoria] = 1
        evento = "NACE la clave → empieza en 1"
    print(f"{vuelta:<7}| {str((categoria, monto)):<22}| {str(conteo_ej):<44}| {evento}")

try:
    print(conteo_ej["snacks"])
except KeyError as error:
    print("\nKeyError: la clave", error, "nunca nació")

### ✍️ Tu turno · Ejercicio 1: conteo por categoría
**Parte A.** `ventas_bodega` tiene tuplas `(fecha, categoria, monto)`. Crea `conteo_por_categoria`: un diccionario `categoría → cantidad de ventas`, con el patrón `if ... in ... / else`. No modifiques `ventas_bodega`.

**Parte B · casos borde.** Predice **sin ejecutar**:

| Variable | Pregunta |
|---|---|
| `pred_in_cero` | ¿qué valor da `"azúcar" in {"azúcar": 0}`? |
| `pred_in_valor` | ¿qué valor da `12 in {"arroz": 12}`? |
| `pred_error` | ¿qué error lanza `{"arroz": 12}["Arroz"]`? Escribe su nombre como texto (por ejemplo, `"ValueError"`). |

In [ ]:
# Tu código aquí

In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Crea el diccionario vacío **antes** del bucle. En cada vuelta pregunta si la categoría ya es clave.
</details>

<details><summary>💡 Pista 2</summary>

`for fecha, categoria, monto in ventas_bodega:` → si `categoria in conteo_por_categoria`, súmale 1; si no, créala **con 1**. En la parte B, recuerda que `in` mira solo las claves y que las mayúsculas cuentan.
</details>

---
## 2. `.get()` para acumular, y los decimales

### 📘 Concepto
`d.get(clave, defecto)` devuelve `d[clave]` si existe y, si no, `defecto`. Nunca lanza `KeyError`. Con él, acumular cabe en una línea y sin `if`:

```python
totales[categoria] = totales.get(categoria, 0) + monto
```

Sin valor por defecto, `.get()` devuelve `None`, y `None + monto` lanza `TypeError`. **Al acumular, pasa siempre el `0`.**

**¿Por qué sale `37.849999999999994`?** Python guarda los decimales en binario y valores como `0.1` no caben exactos, así que al sumar aparecen restos diminutos. No es un error tuyo:
- **Acumula sin redondear y redondea al entregar** el resultado, con `round(x, 2)`.
- Redondear los valores de un diccionario ya armado está permitido: `for c in d: d[c] = round(d[c], 2)`.
- Para comparar decimales no uses `==`: usa `round(x, 2) == y` o `math.isclose(x, y)`.

In [ ]:
# 🔮 Predice antes de ejecutar: ¿el total de "snacks" sale exacto (6.8) o con restos?
# Mi predicción:
import math

ventas_ej = [("lácteos", 12.80), ("snacks", 2.20), ("lácteos", 9.60), ("snacks", 1.10), ("lácteos", 15.45), ("snacks", 3.50)]

totales_ej = {}
for categoria, monto in ventas_ej:
    totales_ej[categoria] = totales_ej.get(categoria, 0) + monto
    print(f"{categoria:<8} + {monto:>6.2f} → {totales_ej}")

print("\nsin redondear:", totales_ej)
print("¿snacks == 6.8?", totales_ej["snacks"] == 6.8, "| ¿isclose?", math.isclose(totales_ej["snacks"], 6.8))
for categoria in totales_ej:
    totales_ej[categoria] = round(totales_ej[categoria], 2)
print("al entregar:  ", totales_ej)

### ✍️ Tu turno · Ejercicio 2: total por categoría con `.get()`
**Parte A.** Con `ventas_bodega`, crea `total_por_categoria`: `categoría → total vendido`. Acumula con `.get(categoria, 0)` (sin `if/else`) y **entrega cada total redondeado a 2 decimales**.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta |
|---|---|
| `pred_igual` | ¿qué valor da `0.1 + 0.2 == 0.3`? |
| `pred_redondeado` | ¿qué valor da `round(0.1 + 0.2, 2) == 0.3`? |
| `pred_error_get` | ¿qué error lanza `{"lunes": 120.0}.get("martes") + 50`? Escribe su nombre como texto. |

In [ ]:
# Tu código aquí

In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Una sola línea dentro del `for`: lee lo que ya hay (o 0) con `.get` y súmale el monto. El redondeo va **después** del `for`.
</details>

<details><summary>💡 Pista 2</summary>

`total_por_categoria[categoria] = total_por_categoria.get(categoria, 0) + monto`. Luego recorre el diccionario y guarda `round(total_por_categoria[categoria], 2)` en la misma clave.
</details>

---
## 3. Recorrer un diccionario y cuándo filtrar el 0

### 📘 Concepto
| Forma | Cada vuelta te da |
|---|---|
| `for k in d` o `for k in d.keys()` | la clave |
| `for v in d.values()` | el valor |
| `for k, v in d.items()` | el par `(clave, valor)` |

- **Cambiar valores** de claves que ya existen mientras recorres está bien.
- **Agregar o quitar claves** mientras recorres lanza `RuntimeError: dictionary changed size during iteration`.

**¿Hay que filtrar los montos 0?** Depende de lo que calculas:
- En un **total o un saldo**, sumar 0 no cambia nada: no hace falta filtrarlo.
- En un **diccionario**, un movimiento de 0 **crea una clave** que no debería existir: ahí sí filtra.
- Para quedarte solo con egresos, el filtro es `monto < 0`, que deja fuera ingresos y ceros a la vez. `monto != 0` deja pasar los ingresos.

In [ ]:
# 🔮 Predice antes de ejecutar: ¿qué claves tendrá `con_distinto_de_cero`? ¿Y `con_menor_que_cero`?
# Mi predicción:

movs_ej = [("sueldo", 900.0), ("mercado", -40.0), ("ajuste", 0.0), ("mercado", -25.5)]

con_distinto_de_cero = {}
con_menor_que_cero = {}
for concepto, monto in movs_ej:
    if monto != 0:
        con_distinto_de_cero[concepto] = con_distinto_de_cero.get(concepto, 0) + monto
    if monto < 0:
        con_menor_que_cero[concepto] = con_menor_que_cero.get(concepto, 0) - monto   # en positivo
print(con_distinto_de_cero)
print(con_menor_que_cero)

for concepto, gasto in con_menor_que_cero.items():
    print(f"{concepto:<8} S/ {gasto:>7.2f}")

try:
    for concepto in con_menor_que_cero:
        con_menor_que_cero["nuevo"] = 1.0      # agregar claves mientras recorres
except RuntimeError as error:
    print("RuntimeError:", error)

### ✍️ Tu turno · Ejercicio 3 🐛: encuentra y arregla el bug
La celda de abajo quiere armar `egresos_por_concepto`: `concepto → gasto`, con **solo los egresos** de `movimientos_junio`, guardados **en positivo** y redondeados a 2 decimales. Corre sin errores, pero el resultado está mal.

Ejecútala, verifica, lee los ❌ y corrígela.

In [ ]:
# 🐛 Este código tiene errores. Corrígelo (puedes reescribirlo).
egresos_por_concepto = {}
for fecha, concepto, monto in movimientos_junio:
    if monto != 0:
        egresos_por_concepto[concepto] = egresos_por_concepto.get(concepto, 0) + monto

In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Hay tres cosas que revisar: **qué** movimientos pasan el filtro, **qué** se suma (mira el signo) y si el resultado se entrega redondeado.
</details>

<details><summary>💡 Pista 2</summary>

El filtro correcto es `monto < 0`. Al acumular, suma `-monto` para que quede en positivo. Al final, redondea recorriendo el diccionario.
</details>

---
## 4. Varios acumuladores en un solo recorrido

### 📘 Concepto
Si necesitas varios resultados de los mismos datos, no escribas un `for` para cada uno: **lleva varios acumuladores en un solo recorrido**.

Lo que depende de un diccionario **ya completo** (un promedio, la categoría con más ventas) va **después**, recorriendo ese diccionario con `.items()`.

**El mayor a mano**, sin `max`: empieza en `None` («todavía no hay ninguno») y reemplázalo cuando encuentres algo mejor:
```python
mejor = None
for clave, valor in d.items():
    if mejor is None or valor > d[mejor]:
        mejor = clave
```

In [ ]:
# 🔮 Predice antes de ejecutar: ¿qué categoría queda como `mas_vendida_ej`?
# Mi predicción:

ventas_ej = [("lácteos", 8.5), ("abarrotes", 23.0), ("lácteos", 6.5), ("bebidas", 4.0), ("abarrotes", 15.0)]

total_ej, cantidad_ej = {}, {}
for categoria, monto in ventas_ej:              # UN recorrido, dos acumuladores
    total_ej[categoria] = total_ej.get(categoria, 0) + monto
    cantidad_ej[categoria] = cantidad_ej.get(categoria, 0) + 1

mas_vendida_ej = None
for categoria, total in total_ej.items():      # después: recorrer el diccionario ya armado
    promedio = total / cantidad_ej[categoria]
    print(f"{categoria:<10} total {total:>6.2f} | {cantidad_ej[categoria]} ventas | promedio {promedio:.2f}")
    if mas_vendida_ej is None or total > total_ej[mas_vendida_ej]:
        mas_vendida_ej = categoria
print("más vendida:", mas_vendida_ej)

### ✍️ Tu turno · Ejercicio 4: compras del mes en un solo recorrido
`compras_mes` tiene las compras a proveedores `(fecha, categoria, monto)`. **Recorriendo `compras_mes` una sola vez**, crea:
1. `compras_total`: `categoría → total comprado`, redondeado a 2 decimales.
2. `compras_cantidad`: `categoría → número de compras`.

Después, recorriendo el diccionario:

3. `compras_promedio`: `categoría → total / cantidad`, redondeado a 2 decimales.
4. `categoria_mas_compras`: la categoría con mayor total, buscada **a mano** con `None` (sin `max`).

In [ ]:
# Tu código aquí

In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Dentro del único `for` sobre `compras_mes` van dos líneas con `.get`: una suma el monto y otra suma 1. El promedio y la categoría mayor salen de un segundo `for` sobre `compras_total.items()`.
</details>

<details><summary>💡 Pista 2</summary>

Calcula el promedio con el total **sin redondear** y redondea el resultado. Para la mayor: `categoria_mas_compras = None` y `if categoria_mas_compras is None or total > compras_total[categoria_mas_compras]:`. Redondea los totales al final.
</details>

---
# Parte 2 · Estado acumulativo y control de flujo
⏱️ 70 minutos · Secciones 5 a 8

---
## 5. El estado arranca en su valor inicial

### 📘 Concepto
- **Agregación final** (`sum`, `len`): un número al final. Responde «¿cuánto quedó?».
- **Seguimiento paso a paso**: guardas el estado en cada vuelta (una lista de saldos). Responde «¿**cuándo** pasó?», «¿**cuántos** días?», «¿cuál fue el **peor** momento?».

El estado **nace con el valor que tenía antes del primer movimiento**:
```python
saldo = saldo_inicial        # no 0
for fecha, concepto, monto in movimientos:
    saldo += monto
    saldos.append(saldo)
```
Si arrancas en 0 y sumas `saldo_inicial` al final, el saldo final coincide, y por eso parece correcto. Pero **cada saldo intermedio está desplazado**, y todo lo que preguntes sobre el camino (¿estuvo en rojo?, ¿cuál fue el mínimo?) sale mal.

In [ ]:
# 🔮 Predice antes de ejecutar: ¿en qué vueltas dicen cosas distintas las dos columnas "¿rojo?"?
# Mi predicción:

movimientos_ej = [
    ("2026-02-01", "ventas", 45.00),
    ("2026-02-02", "proveedor", -120.00),
    ("2026-02-03", "ventas", 10.00),
    ("2026-02-04", "ajuste", 0.00),
    ("2026-02-05", "ventas", 40.00),
    ("2026-02-06", "luz", -40.00),
    ("2026-02-07", "ventas", 55.00),
    ("2026-02-08", "agua", -25.00),
]
saldo_inicial_ej = 60.0

saldo_mal = 0                   # arranca en 0 (bug)
saldo_bien = saldo_inicial_ej   # arranca en el valor inicial
print(f"{'vuelta':<7}| {'monto':>8} | {'saldo_mal':>9} | {'saldo_bien':>10} | ¿rojo (mal)? | ¿rojo (bien)?")
for vuelta, (fecha, concepto, monto) in enumerate(movimientos_ej, start=1):
    saldo_mal += monto
    saldo_bien += monto
    print(f"{vuelta:<7}| {monto:>8.2f} | {saldo_mal:>9.2f} | {saldo_bien:>10.2f} | {str(saldo_mal < 0):<12} | {saldo_bien < 0}")

print("\nsaldo final (mal + inicial):", saldo_mal + saldo_inicial_ej, "| saldo final (bien):", saldo_bien)

**Gráfico ilustrativo** (aquí sí usamos `matplotlib`): el saldo día a día, la línea del 0 y los días en sobregiro marcados.

In [ ]:
import matplotlib.pyplot as plt

movimientos_ej = [
    ("2026-02-01", "ventas", 45.00),
    ("2026-02-02", "proveedor", -120.00),
    ("2026-02-03", "ventas", 10.00),
    ("2026-02-04", "ajuste", 0.00),
    ("2026-02-05", "ventas", 40.00),
    ("2026-02-06", "luz", -40.00),
    ("2026-02-07", "ventas", 55.00),
    ("2026-02-08", "agua", -25.00),
]
saldo_inicial_ej = 60.0

dias_ej, saldos_ej, dias_rojos_ej, saldos_rojos_ej = [], [], [], []
saldo = saldo_inicial_ej
for fecha, concepto, monto in movimientos_ej:
    saldo += monto
    dias_ej.append(fecha[5:])                  # "02-01"
    saldos_ej.append(saldo)
    if saldo < 0:
        dias_rojos_ej.append(fecha[5:])
        saldos_rojos_ej.append(saldo)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.axhspan(min(saldos_ej) - 15, 0, color="#c0392b", alpha=0.07)       # zona en rojo
ax.plot(dias_ej, saldos_ej, color="#2a6fbb", linewidth=2, marker="o", markersize=6, label="Saldo al cierre del día")
ax.scatter(dias_rojos_ej, saldos_rojos_ej, color="#c0392b", s=90, zorder=3, label="Día en sobregiro")
ax.axhline(0, color="#444444", linewidth=1.2)                          # línea del 0
ax.set_title("Saldo de la cuenta día a día (S/)")
ax.set_xlabel("Día (mes-día)")
ax.set_ylabel("Saldo (S/)")
ax.grid(axis="y", color="#dddddd", linewidth=0.8)
ax.legend(frameon=False, loc="upper right")
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.show()

### ✍️ Tu turno · Ejercicio 5: saldos paso a paso
La caja de una bodega abre con `saldo_inicial_caja` y registra `movimientos_caja` `(fecha, concepto, monto)`. Crea:
1. `saldos_caja`: lista con el saldo **después de cada movimiento**, cada uno redondeado a 2 decimales (uno por movimiento, también el del ajuste de 0).
2. `saldo_cierre`: el último saldo, redondeado a 2 decimales.

In [ ]:
# Tu código aquí

In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

Necesitas una variable de estado (el saldo) y una lista vacía. En cada vuelta: actualiza el estado y guarda una copia redondeada en la lista.
</details>

<details><summary>💡 Pista 2</summary>

`saldo = saldo_inicial_caja` (¡no 0!), `saldos_caja = []`; dentro del `for`: `saldo += monto` y `saldos_caja.append(round(saldo, 2))`. No filtres el monto 0: ese día también tiene saldo.
</details>

---
## 6. Eventos dentro del bucle: el primero, todos y los cruces

### 📘 Concepto
Tres preguntas distintas sobre el mismo recorrido:

| Pregunta | Patrón |
|---|---|
| El **primer** día en rojo | `primero = None` antes; al encontrarlo, guarda la fecha y `break` |
| **Todos** los días en rojo | `lista.append(fecha)` en cada vuelta con `saldo < 0` |
| **Cuántas veces entra** en rojo | recuerda `saldo_anterior` y cuenta cuando `saldo_anterior >= 0 and saldo < 0` |

- **Estar** en rojo es un estado: basta el saldo actual.
- **Entrar** en rojo es un evento: necesitas el saldo **anterior**. Actualízalo **al final** de cada vuelta, después de usarlo.

**`break` vs `continue`:** `break` termina el bucle entero. `continue` salta al siguiente elemento, y todo lo que hay **debajo** de él no se ejecuta en esa vuelta. Un `continue` para «saltar los montos 0», puesto **antes** del chequeo de sobregiro, se salta un día que **sí** estaba en rojo.

In [ ]:
# 🔮 Predice antes de ejecutar: ¿cuántos días en rojo hay? ¿Cuántas veces ENTRA la cuenta en rojo?
# Mi predicción:

movimientos_ej = [
    ("2026-02-01", "ventas", 45.00),
    ("2026-02-02", "proveedor", -120.00),
    ("2026-02-03", "ventas", 10.00),
    ("2026-02-04", "ajuste", 0.00),
    ("2026-02-05", "ventas", 40.00),
    ("2026-02-06", "luz", -40.00),
    ("2026-02-07", "ventas", 55.00),
    ("2026-02-08", "agua", -25.00),
]
saldo_inicial_ej = 60.0

saldo = saldo_inicial_ej
saldo_anterior = saldo
entradas_ej = 0
print(f"{'vuelta':<7}| {'fecha':<11}| {'monto':>8} | {'anterior → nuevo':<19}| evento")
for vuelta, (fecha, concepto, monto) in enumerate(movimientos_ej, start=1):
    saldo += monto
    if saldo_anterior >= 0 and saldo < 0:
        evento = "ENTRA en rojo"
        entradas_ej += 1
    elif saldo_anterior < 0 and saldo >= 0:
        evento = "SALE del rojo"
    elif saldo < 0:
        evento = "sigue en rojo"
    else:
        evento = "-"
    print(f"{vuelta:<7}| {fecha:<11}| {monto:>8.2f} | {saldo_anterior:>7.2f} → {saldo:>7.2f} | {evento}")
    saldo_anterior = saldo          # se actualiza AL FINAL de la vuelta
print("\nentradas en rojo:", entradas_ej)

In [ ]:
# 🔮 Predice antes de ejecutar: ¿cuántos días en rojo cuenta cada versión?
# Mi predicción:

movimientos_ej = [
    ("2026-02-01", "ventas", 45.00),
    ("2026-02-02", "proveedor", -120.00),
    ("2026-02-03", "ventas", 10.00),
    ("2026-02-04", "ajuste", 0.00),
    ("2026-02-05", "ventas", 40.00),
    ("2026-02-06", "luz", -40.00),
    ("2026-02-07", "ventas", 55.00),
    ("2026-02-08", "agua", -25.00),
]
saldo_inicial_ej = 60.0

for version in ("con continue antes del chequeo", "sin continue (correcta)"):
    saldo = saldo_inicial_ej
    dias_rojo_ej = []
    for fecha, concepto, monto in movimientos_ej:
        if version.startswith("con") and monto == 0:
            continue                          # se salta TODO lo de abajo en esta vuelta
        saldo += monto
        if saldo < 0:
            dias_rojo_ej.append(fecha)
    print(f"{version:<31} → {dias_rojo_ej}")

### ✍️ Tu turno · Ejercicio 6: sobregiro, el primero, todos y las entradas
Con `saldo_inicial_cuenta` y `movimientos_cuenta`, crea:
1. `primer_rojo`: la fecha del **primer** día en que el saldo queda por debajo de 0. Usa `break`.
2. `dias_en_rojo`: lista con **todas** las fechas en que el saldo, después del movimiento, quedó por debajo de 0.
3. `entradas_en_rojo`: cuántas veces el saldo **pasó** de ≥ 0 a < 0.
4. `primer_rojo_ahorro`: lo mismo que el punto 1, pero con `saldo_inicial_ahorro` y `movimientos_ahorro`. Esa cuenta nunca queda en rojo, así que el resultado debe ser `None`.

El recorrido con `break` tiene que ir aparte, porque corta el bucle. Los puntos 2 y 3 caben en un solo recorrido.

In [ ]:
# Tu código aquí

In [ ]:
# ✅ Verificar
check_ejercicio_6()

<details><summary>💡 Pista 1</summary>

Para el punto 1 empieza con `primer_rojo = None`. Cada recorrido reinicia su saldo con **su** saldo inicial. Para el 3 necesitas `saldo_anterior`, que arranca igual que el saldo.
</details>

<details><summary>💡 Pista 2</summary>

Dentro del `for` de los puntos 2 y 3: `saldo += monto`; `if saldo < 0:` agrega la fecha y, si además `saldo_anterior >= 0`, suma una entrada; en la **última** línea de la vuelta, `saldo_anterior = saldo`. No pongas `continue` con el monto 0.
</details>

---
## 7. Extremos a mano: `None` como centinela

### 📘 Concepto
Un **centinela** es un valor que significa «todavía no hay ninguno». `None` es el correcto porque no puede confundirse con un dato real.

```python
mejor_valor, mejor_registro = None, None
for registro in registros:
    if mejor_valor is None or registro[2] < mejor_valor:     # `is None`, no `== None`
        mejor_valor, mejor_registro = registro[2], registro   # guarda el registro COMPLETO
```

- **Inicializar en 0 es un bug latente.** Para el egreso más grande funciona, porque el 0 hace de filtro. Pero si todos los valores son positivos (la venta más pequeña, el saldo mínimo de una cuenta que nunca baja de 0), el 0 «gana» y nunca se actualiza.
- **Guarda el registro completo**, no solo el valor: así tienes la fecha sin recorrer otra vez.
- **Empates:** con `<` te quedas con el **primero** que alcanzó el extremo; con `<=`, con el **último**.

In [ ]:
# 🔮 Predice antes de ejecutar: ¿qué imprime cada versión en cada lista?
# Mi predicción:

listas_ej = {
    "con egresos": [("2026-04-01", 120.0), ("2026-04-02", -35.0), ("2026-04-03", -80.0)],
    "solo ventas": [("2026-04-01", 38.5), ("2026-04-02", 52.0), ("2026-04-03", 29.9)],
}

for nombre, registros in listas_ej.items():
    menor_cero = 0                           # centinela equivocado
    menor_none, registro_none = None, None   # centinela correcto
    for registro in registros:
        if registro[1] < menor_cero:
            menor_cero = registro[1]
        if menor_none is None or registro[1] < menor_none:
            menor_none, registro_none = registro[1], registro
    print(f"{nombre:<12} | empezando en 0 → {menor_cero:<6} | empezando en None → {registro_none}")

### ✍️ Tu turno · Ejercicio 7 🐛: el saldo mínimo
**Parte A.** La celda de abajo busca el **saldo más bajo** de agosto (después de cada movimiento) y la **fecha** en que ocurrió. Si hay empate, vale la primera fecha. Crea o corrige `saldo_minimo` (redondeado a 2 decimales) y `fecha_saldo_minimo`. El código corre sin errores, pero el resultado está mal.

**Parte B.** Con esta lista, predice **sin ejecutar** el nombre del proveedor que queda como «más barato» en cada caso:

```python
precios_proveedor = [("Proveedor A", 12.5), ("Proveedor B", 11.9), ("Proveedor C", 11.9), ("Proveedor D", 13.0)]
mejor = None
for p in precios_proveedor:
    if mejor is None or p[1] < mejor[1]:      # en la otra versión: p[1] <= mejor[1]
        mejor = p
```

| Variable | Pregunta |
|---|---|
| `pred_con_menor` | ¿`mejor[0]` con `<`? |
| `pred_con_menor_igual` | ¿`mejor[0]` con `<=`? |

In [ ]:
# 🐛 Parte A: este código tiene un error típico. Corrígelo (puedes reescribirlo).
saldo = saldo_inicial_agosto
saldo_minimo = 0
fecha_saldo_minimo = None
for fecha, concepto, monto in movimientos_agosto:
    saldo += monto
    if saldo < saldo_minimo:
        saldo_minimo = saldo
        fecha_saldo_minimo = fecha

# Parte B: tus predicciones

In [ ]:
# ✅ Verificar
check_ejercicio_7()

<details><summary>💡 Pista 1</summary>

¿Qué pasa con el valor inicial del mínimo si **ningún** saldo es menor que él? ¿Quién es «el mejor hasta ahora» antes de mirar el primer movimiento?
</details>

<details><summary>💡 Pista 2</summary>

Cambia `saldo_minimo = 0` por `None` y la condición por `saldo_minimo is None or saldo < saldo_minimo`. Mantén `<` para quedarte con la primera fecha y redondea al final.
</details>

---
## 8. Todo junto: un kárdex en un solo recorrido

### 📘 Concepto
Antes de escribir el bucle, **haz la lista de tus variables de estado y su valor inicial**. Es la mitad del ejercicio:

| Qué quiero saber | Variable | Empieza en |
|---|---|---|
| el valor actual | `stock` | `stock_inicial` |
| todas las veces que pasa algo | una lista | `[]` |
| la primera vez que pasa algo | `fecha_...` | `None` |
| un cruce (antes sí, ahora no) | `stock_anterior` | se guarda al inicio de cada vuelta |
| el extremo de un subconjunto | `mayor_...` | `None` |

Dentro del bucle, cada variable se actualiza en su momento, y el orden importa.

### ✍️ Tu turno · Ejercicio 8: kárdex del aceite
`kardex` tiene los movimientos del aceite de 1 L como `(fecha, tipo, cantidad)`, donde `tipo` es `"entrada"` (suma) o `"salida"` (resta). El stock arranca en `stock_inicial` y el mínimo aceptable es `stock_minimo`. **En un solo recorrido**, crea:
1. `stock_final`: el stock después del último movimiento.
2. `fechas_bajo_minimo`: lista con las fechas en que el stock, después del movimiento, quedó **por debajo** de `stock_minimo`.
3. `fecha_recuperacion`: la **primera** fecha en que el stock pasa de estar bajo el mínimo a **superarlo** (`> stock_minimo`). `None` si nunca ocurre.
4. `mayor_salida`: la tupla completa de la **salida** con más unidades. Si hay empate, la primera.

In [ ]:
# Tu código aquí

In [ ]:
# ✅ Verificar
check_ejercicio_8()

<details><summary>💡 Pista 1</summary>

Variables de estado: `stock` (empieza en `stock_inicial`), `fechas_bajo_minimo = []`, `fecha_recuperacion = None`, `mayor_salida = None`. Para la recuperación guarda `stock_anterior = stock` al **inicio** de cada vuelta.
</details>

<details><summary>💡 Pista 2</summary>

Si el tipo es salida, resta y compara con la mayor salida (`mayor_salida is None or cantidad > mayor_salida[2]`). Luego: `if stock < stock_minimo` agrega la fecha; `if fecha_recuperacion is None and stock_anterior < stock_minimo and stock > stock_minimo` guarda la fecha.
</details>

---
# Parte 3 · Comparación de tuplas, `key`, `zip` y tablas
⏱️ 50 minutos · Secciones 9 a 12

---
## 9. Cómo compara Python tuplas y textos

### 📘 Concepto
**Tuplas:** se comparan **elemento por elemento, de izquierda a derecha**.
1. Compara el índice 0. Si son distintos, ahí se decide y **nada más se mira**.
2. Solo si empatan, pasa al índice 1, y así sucesivamente.
3. Si todo lo comparado es igual, la tupla más corta es la menor.

**Textos:** se comparan **carácter por carácter**, no como números: `"10" < "9"` es `True` porque `"1"` va antes que `"9"`.

Por eso las fechas ISO **con ceros** (`"2026-09-02"`, todas del mismo largo) ordenan bien como texto, y **sin ceros** (`"2026-9-10"`) no.

In [ ]:
# 🔮 Predice antes de ejecutar: ¿True o False en la primera línea? ¿Qué orden sale en las dos últimas?
# Mi predicción:

a_ej = ("2026-09-10", -5)
b_ej = ("2026-09-02", -500)
for i in range(len(a_ej)):
    if a_ej[i] == b_ej[i]:
        print(f"índice {i}: {a_ej[i]!r} == {b_ej[i]!r} → empate, sigo")
    else:
        print(f"índice {i}: {a_ej[i]!r} vs {b_ej[i]!r} → decide aquí: {a_ej[i] < b_ej[i]}")
        break
print("a_ej < b_ej:", a_ej < b_ej)

print(sorted(["2026-10-01", "2026-09-15", "2026-09-02"]), "← con ceros: bien")
print(sorted(["2026-10-1", "2026-9-15", "2026-9-2"]), "← sin ceros: orden roto")

### ✍️ Tu turno · Ejercicio 9: predicciones de comparación
Predice **sin ejecutar** (`True` o `False`; en `pred_min`, la tupla que devuelve):

| Variable | Expresión |
|---|---|
| `pred_a` | `("2026-11-03", 10) < ("2026-11-03", 9)` |
| `pred_b` | `("b", 1) < ("a", 99)` |
| `pred_c` | `(1, 500) < (1, 1000)` |
| `pred_d` | `("2026-10-5", 0) < ("2026-10-15", 0)` |
| `pred_e` | `"100" < "25"` |
| `pred_f` | `(7, 3) < (7, 3, 0)` |
| `pred_min` | `min(lista_prueba)` (mira `lista_prueba` en 📦 Tus datos) |

In [ ]:
# Tu código aquí

In [ ]:
# ✅ Verificar
check_ejercicio_9()

<details><summary>💡 Pista 1</summary>

En cada par de tuplas mira **solo el índice 0**. Si son distintos, ahí se decide. Si empatan, pasa al índice 1.
</details>

<details><summary>💡 Pista 2</summary>

En los textos compara carácter por carácter desde la izquierda. Para `pred_min`, busca la fecha más antigua y, si hay dos con esa fecha, mira el monto.
</details>

---
## 10. La trampa de `min(lista_de_tuplas)` y el parámetro `key`

### 📘 Concepto
`min(movimientos)` compara **tuplas completas**, así que decide por el **índice 0, la fecha**: devuelve el movimiento más **antiguo**, no el de menor monto. `max` devuelve el más **reciente**. No da error: da un resultado con pinta razonable, pero incorrecto.

`key` le dice a Python **qué comparar**. Recibe una función que se aplica a cada elemento y devuelve el elemento **original**:

| Quiero | Código |
|---|---|
| movimiento de menor monto | `min(movs, key=lambda m: m[2])` |
| movimiento de mayor monto | `max(movs, key=lambda m: m[2])` |
| ordenar por monto, de menor a mayor | `sorted(movs, key=lambda m: m[2])` |
| ordenar de mayor a menor | `sorted(movs, key=lambda m: m[2], reverse=True)` |
| mayor monto y, si empatan, fecha más antigua | `sorted(movs, key=lambda m: (-m[2], m[0]))` |
| la clave con mayor valor de un diccionario | `max(d, key=d.get)` |

`lambda m: m[2]` es una función pequeña sin nombre: «dado `m`, devuelve `m[2]`». Si varios empatan en la `key`, `min` y `max` devuelven el **primero**.

⚠️ Con montos negativos, «el egreso más grande» es el número **más bajo**: `sorted(..., key=lambda m: m[2])` ya lo deja primero, y `reverse=True` lo manda al final.

In [ ]:
# 🔮 Predice antes de ejecutar: ¿qué devuelve `min` sin key? ¿Y con key?
# Mi predicción:

movs_ej = [
    ("2026-03-01", "mercado", -45.20),
    ("2026-03-06", "sueldo", 1800.00),
    ("2026-03-09", "alquiler", -650.00),
    ("2026-03-14", "yape", 230.00),
    ("2026-03-20", "luz", -96.40),
]
print("min sin key :", min(movs_ej), "← el más ANTIGUO")
print("min con key :", min(movs_ej, key=lambda m: m[2]))
print("max con key :", max(movs_ej, key=lambda m: m[2]))
print("2 egresos más grandes:", sorted(movs_ej, key=lambda m: m[2])[:2])

gasto_ej = {"abarrotes": 320.5, "lácteos": 185.0, "bebidas": 402.25}
print("clave con mayor valor:", max(gasto_ej, key=gasto_ej.get))
print("ranking:", sorted(gasto_ej.items(), key=lambda kv: kv[1], reverse=True))

### ✍️ Tu turno · Ejercicio 10 🐛: mayor ingreso, mayor egreso y top 3
La celda de abajo quiere obtener, de `movimientos_abril`:
1. `mayor_ingreso`: la tupla completa del movimiento con el monto más alto.
2. `mayor_egreso_abril`: la tupla completa del movimiento con el monto más negativo.
3. `top3_egresos`: lista con los 3 egresos más grandes, **del más negativo al menos negativo**.

Corre sin errores, pero los tres resultados están mal. Corrígela.

In [ ]:
# 🐛 Este código tiene errores. Corrígelo (puedes reescribirlo).
mayor_ingreso = max(movimientos_abril)
mayor_egreso_abril = min(movimientos_abril)

egresos_abril = []
for movimiento in movimientos_abril:
    if movimiento[2] < 0:
        egresos_abril.append(movimiento)
top3_egresos = sorted(egresos_abril, reverse=True)[:3]

In [ ]:
# ✅ Verificar
check_ejercicio_10()

<details><summary>💡 Pista 1</summary>

Sin `key`, `max`, `min` y `sorted` comparan la tupla completa, empezando por el índice 0. ¿Qué hay en el índice 0?
</details>

<details><summary>💡 Pista 2</summary>

Agrega `key=lambda m: m[2]` en los tres. Para el top 3, como los egresos son negativos, el orden ascendente ya deja primero el más negativo: quita `reverse=True`.
</details>

---
## 11. `zip`, `enumerate` y `dict(zip(...))`

### 📘 Concepto
| Herramienta | Qué da | Ojo |
|---|---|---|
| `zip(a, b)` | pares `(a[0], b[0])`, `(a[1], b[1])`… de listas paralelas | se corta en la más corta, sin avisar |
| `enumerate(a, start=1)` | pares `(1, a[0])`, `(2, a[1])`… | sin `start=1` cuenta desde 0 |
| `dict(zip(claves, valores))` | un diccionario en **una línea** | si hay claves repetidas, gana la última |

Combinados: `for numero, (fecha, venta) in enumerate(zip(fechas, ventas), start=1):` (el par de `zip` va entre paréntesis).

Para un ranking con desempate, arma primero una lista de tuplas con todo lo que necesitas y ordénala con una `key` que devuelva una tupla: `key=lambda r: (-r[2], r[3])` ordena por el índice 2 de mayor a menor y, si empatan, por el índice 3 de menor a mayor.

In [ ]:
# 🔮 Predice antes de ejecutar: ¿cuántos pares da zip? ¿Qué día gana el ranking y por qué?
# Mi predicción:

dias_ej = ["lunes", "martes", "miércoles", "jueves"]
ventas_ej = [120.0, 210.0, 95.5, 210.0, 80.0]      # ¡una venta de más!
clientes_ej = [30, 52, 25, 41]

print(list(zip(dias_ej, ventas_ej)), "← se cortó en la más corta")
print(dict(zip(dias_ej, ventas_ej)))

registros_ej = []
for numero, (dia, venta, clientes) in enumerate(zip(dias_ej, ventas_ej, clientes_ej), start=1):
    registros_ej.append((numero, dia, venta, clientes))
for puesto, registro in enumerate(sorted(registros_ej, key=lambda r: (-r[2], r[3])), start=1):
    print(f"{puesto}.º", registro)

### ✍️ Tu turno · Ejercicio 11: ranking de días con desempate
`fechas_semana`, `ventas_dia` y `clientes_dia` son listas paralelas de una semana. Crea:
1. `venta_por_fecha`: diccionario `fecha → venta` en **una línea** con `dict(zip(...))`.
2. `ranking`: lista de tuplas `(numero_dia, fecha, venta, clientes)`, donde `numero_dia` va de 1 a 7. Ordénala de **mayor a menor venta** y, si dos días empatan en venta, primero el de **menos clientes**. Usa `enumerate(zip(...), start=1)`.
3. `dia_ganador` y `fecha_ganadora`: el número de día y la fecha del primer lugar.

In [ ]:
# Tu código aquí

In [ ]:
# ✅ Verificar
check_ejercicio_11()

<details><summary>💡 Pista 1</summary>

Primero arma una lista con `append` recorriendo `enumerate(zip(fechas_semana, ventas_dia, clientes_dia), start=1)`. Después ordénala.
</details>

<details><summary>💡 Pista 2</summary>

`ranking = sorted(registros, key=lambda r: (-r[2], r[3]))`: el signo menos invierte el orden de la venta y `r[3]` desempata. El ganador es `ranking[0]`.
</details>

---
## 12. Tablas: el mejor de cada fila

### 📘 Concepto
En una lista de listas, `tabla[i]` es la fila `i` y `tabla[i][j]` el valor de la fila `i`, columna `j`. Para un resultado **por fila**, recorre las filas y, dentro, trabaja con cada una:

```python
for nombre, fila in zip(nombres, tabla):         # una vuelta por fila
    mejor_j = 0
    for j in range(1, len(fila)):                 # un bucle dentro de otro
        if fila[j] > fila[mejor_j]:               # `>`: en el empate gana la primera columna
            mejor_j = j
```

Atajo: `fila.index(max(fila))` da la posición de la **primera** aparición del máximo. Las posiciones empiezan en 0: si te piden «mes 1 a 6», suma 1.

In [ ]:
# 🔮 Predice antes de ejecutar: ¿en qué día (1 a 3) tuvo su mejor venta cada vendedor?
# Mi predicción:

vendedores_ej = ["Ana", "Luis"]
tabla_ej = [
    [120, 340, 340],     # Ana: empate entre el día 2 y el 3
    [200, 150, 310],     # Luis
]

for nombre, fila in zip(vendedores_ej, tabla_ej):
    mejor_j = 0
    for j in range(1, len(fila)):
        if fila[j] > fila[mejor_j]:
            mejor_j = j
    con_index = fila.index(max(fila))
    print(f"{nombre:<5} {fila} → mejor día {mejor_j + 1} (con .index: {con_index + 1}) | total {sum(fila)}")

### ✍️ Tu turno · Ejercicio 12: el mejor mes de cada local
En `ventas_mensuales` cada fila es un local (en el orden de `locales`) y cada columna un mes (en el orden de `meses`, de enero a junio). Crea:
1. `total_por_local`: diccionario `local → venta total de sus 6 meses`.
2. `mejor_mes_por_local`: diccionario `local → número de mes (1 a 6)` en que tuvo su mayor venta. Si hay empate, el primer mes.
3. `local_top`: el local con mayor venta total.

In [ ]:
# Tu código aquí

In [ ]:
# ✅ Verificar
check_ejercicio_12()

<details><summary>💡 Pista 1</summary>

Recorre `zip(locales, ventas_mensuales)`: en cada vuelta tienes el nombre y su fila completa. El total es `sum(fila)`.
</details>

<details><summary>💡 Pista 2</summary>

Para el mejor mes usa un bucle interno con `>` (o `fila.index(max(fila))`) y súmale 1. Para `local_top`: `max(total_por_local, key=total_por_local.get)` o una búsqueda a mano con `None`.
</details>

---
## 🏋️ Reto final: el extracto de octubre
⏱️ 40 minutos

`movimientos` es una lista de tuplas `(fecha, concepto, monto)`, una por día y en orden: los montos positivos son ingresos y los negativos, egresos. `saldo_inicial` es el saldo antes del primer movimiento. Hay un movimiento con monto 0 (un ajuste): no es ingreso ni egreso.

⚠️ Todas las trampas del notebook están activas: el ajuste de 0 cae en un día en sobregiro, hay **un empate** en el mayor egreso, el movimiento más antiguo **no** es el mayor egreso, hay conceptos con un solo egreso y la cuenta **entra dos veces** en sobregiro.

Crea (no modifiques `movimientos`):

| Variable | Qué es |
|---|---|
| `saldo_final` | saldo después del último movimiento, redondeado a 2 decimales |
| `n_ingresos` / `n_egresos` | cantidad de montos mayores que 0 / menores que 0 |
| `gasto_por_concepto` | `concepto → total gastado`, solo egresos, **en positivo**, cada total redondeado a 2 decimales |
| `mayor_egreso` | la tupla completa del movimiento con el monto más negativo; si hay empate, **el primero** |
| `dias_sobregiro` | lista de fechas en que el saldo, **después** del movimiento, quedó por debajo de 0 |
| `veces_en_sobregiro` | cuántas veces el saldo **pasó** de ≥ 0 a < 0 |
| `primer_dia_sobregiro` | la primera fecha con saldo < 0 (`None` si nunca pasa) |
| `concepto_mas_gasto` | el concepto con mayor gasto total |

#### ✍️ Tu turno · Reto · Parte A: resuélvelo a tu manera
Usa tantos recorridos como necesites: lo importante es que cada resultado sea correcto.

In [ ]:
# Tu código aquí

In [ ]:
# ✅ Verificar
check_reto("A")

<details><summary>💡 Pista 1</summary>

Piensa qué tipo de pregunta es cada resultado: un **total** (recorrido completo), un **estado** (el saldo, que arranca en `saldo_inicial`), un **evento** (primero, todos, entradas con `saldo_anterior`) o un **extremo** (`None` o `key`).
</details>

<details><summary>💡 Pista 2</summary>

Trampas: no filtres el monto 0 al calcular el saldo ni pongas un `continue` antes del chequeo de sobregiro; para el mayor egreso usa `key` o `<` (no `<=`); `saldo_anterior = saldo` va al final de la vuelta; redondea al entregar.
</details>

#### Parte B · Ahora, en un solo `for`
Rehaz **el mismo reto** con **un único recorrido** de `movimientos` que lleve todos los acumuladores a la vez. Solo lo que depende del diccionario **ya completo** (redondear el gasto y buscar el concepto con más gasto) puede ir después del `for`. Mismas variables, mismo verificador.

#### ✍️ Tu turno · Reto · Parte B: un solo recorrido
Declara todos los acumuladores **antes** del `for`, actualízalos dentro y entrega después.

In [ ]:
# Tu código aquí

In [ ]:
# ✅ Verificar
check_reto("B")

<details><summary>💡 Pista 1</summary>

Antes del `for`: `saldo`, `saldo_anterior`, los dos contadores, el diccionario, `mayor_egreso = None`, la lista, el contador de entradas y `primer_dia_sobregiro = None`.
</details>

<details><summary>💡 Pista 2</summary>

Dentro: `saldo += monto`; `if monto > 0` / `elif monto < 0` (ahí el gasto y el mayor egreso); luego `if saldo < 0` (días, entradas, primer día); y en la **última** línea, `saldo_anterior = saldo`.
</details>

---
## 🚀 Nivel pro (opcional)
Con los datos del reto:
1. `saldo_por_fecha`: diccionario `fecha → saldo después de ese movimiento`, redondeado a 2 decimales. Arma primero una lista de fechas y otra de saldos, y construye el diccionario en **una línea** con `dict(zip(...))`.
2. `ranking_conceptos`: lista de tuplas `(concepto, gasto)` ordenada de **mayor a menor gasto** y, si empatan, por nombre en orden alfabético. Parte de un diccionario de gasto por concepto y usa `sorted(....items(), key=...)`.
3. `racha_mas_larga`: la mayor cantidad de movimientos **seguidos** con el saldo en rojo. Pista: un contador que sube en rojo y vuelve a 0 al salir, y otro que guarda el mejor.

#### ✍️ Tu turno · Nivel pro
Estos tres ejercicios no tienen pistas: combinan lo que ya practicaste.

In [ ]:
# Tu código aquí

In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Explicar por qué `d[clave]` da `KeyError` y cuándo usar `in` o `.get(clave, 0)`.
- [ ] Agrupar (conteo, total y promedio) con varios acumuladores en un solo `for`.
- [ ] Redondear al entregar y comparar decimales sin `==`.
- [ ] Explicar por qué el saldo arranca en `saldo_inicial` y no en 0.
- [ ] Distinguir «estar en rojo» de «entrar en rojo», y saber qué variable necesita lo segundo.
- [ ] Buscar un extremo con `None`, guardar el registro completo y decidir el empate con `<` o `<=`.
- [ ] Saber cuándo un `continue` es seguro y cuándo se salta un caso válido.
- [ ] Predecir comparaciones de tuplas y de textos, y evitar la trampa de `min(lista_de_tuplas)`.
- [ ] Usar `key` en `min`, `max` y `sorted`, incluida una `key` que devuelve una tupla.
- [ ] Combinar `zip`, `enumerate(..., start=1)` y `dict(zip(...))`, y encontrar el mejor de cada fila de una tabla.

### Lo que viene: los mismos patrones con otras herramientas
| Lo que practicaste hoy | En SQL | En pandas |
|---|---|---|
| Agrupar en un diccionario (`d[k] = d.get(k, 0) + x`) | `GROUP BY` | `groupby` |
| Saldo acumulado paso a paso | `SUM() OVER (ORDER BY fecha)` | `cumsum` |
| `min(registros, key=...)` para hallar la fila del mínimo | `ORDER BY ... LIMIT 1` | `idxmin` |
| Varios acumuladores en un solo recorrido | varias agregaciones en un `SELECT` | `agg` |

Esas herramientas harán por ti el recorrido que hoy escribiste a mano. Saber lo que pasa por dentro te ayudará a detectar resultados raros.

---
## 📎 Anexo · Soluciones
Ábrelas **solo después de intentarlo** y de leer lo que te dice ✅ Verificar. Cada solución usa los datos del setup: si quieres ejecutarla, cópiala en la celda de su ejercicio.

<details><summary>🔒 Ejercicio 1: conteo por categoría</summary>

```python
# Parte A
conteo_por_categoria = {}
for fecha, categoria, monto in ventas_bodega:
    if categoria in conteo_por_categoria:
        conteo_por_categoria[categoria] += 1   # ya existía: acumular
    else:
        conteo_por_categoria[categoria] = 1    # primera vez: nace con 1

# Parte B
pred_in_cero = True       # la clave existe, aunque su valor sea 0
pred_in_valor = False     # `in` busca entre las claves, no entre los valores
pred_error = "KeyError"   # "Arroz" y "arroz" son claves distintas
```

**Error típico:** inicializar en 0 dentro del `else` hace que la primera venta de cada categoría no cuente. Sumar `monto` en lugar de 1 convierte el conteo en un total.
</details>

<details><summary>🔒 Ejercicio 2: total por categoría con `.get()`</summary>

```python
# Parte A
total_por_categoria = {}
for fecha, categoria, monto in ventas_bodega:
    total_por_categoria[categoria] = total_por_categoria.get(categoria, 0) + monto

for categoria in total_por_categoria:          # al entregar: redondear
    total_por_categoria[categoria] = round(total_por_categoria[categoria], 2)

# Parte B
pred_igual = False          # 0.1 + 0.2 da 0.30000000000000004
pred_redondeado = True
pred_error_get = "TypeError"   # .get sin defecto devuelve None, y None + 50 falla
```

**Error típico:** usar `.get(categoria)` sin el 0 (da `None` y luego `TypeError`) o entregar los totales sin redondear. Recorrer otra vez las ventas para redondear no hace falta: basta recorrer el diccionario.
</details>

<details><summary>🔒 Ejercicio 3 🐛: encuentra y arregla el bug</summary>

```python
egresos_por_concepto = {}
for fecha, concepto, monto in movimientos_junio:
    if monto < 0:                                # solo egresos: fuera ingresos y ceros
        egresos_por_concepto[concepto] = egresos_por_concepto.get(concepto, 0) - monto   # en positivo

for concepto in egresos_por_concepto:            # al entregar: redondear
    egresos_por_concepto[concepto] = round(egresos_por_concepto[concepto], 2)
```

**Error típico:** `monto != 0` deja pasar los ingresos y mezcla signos. Con `monto < 0` se descartan ingresos y ceros de una vez, y sumando `-monto` el gasto queda en positivo.
</details>

<details><summary>🔒 Ejercicio 4: compras del mes en un solo recorrido</summary>

```python
compras_total = {}
compras_cantidad = {}
for fecha, categoria, monto in compras_mes:        # un solo recorrido, dos acumuladores
    compras_total[categoria] = compras_total.get(categoria, 0) + monto
    compras_cantidad[categoria] = compras_cantidad.get(categoria, 0) + 1

compras_promedio = {}
categoria_mas_compras = None                       # centinela: todavía no hay ninguna
for categoria, total in compras_total.items():
    compras_promedio[categoria] = round(total / compras_cantidad[categoria], 2)
    if categoria_mas_compras is None or total > compras_total[categoria_mas_compras]:
        categoria_mas_compras = categoria

for categoria in compras_total:                    # al entregar: redondear
    compras_total[categoria] = round(compras_total[categoria], 2)
```

**Error típico:** comparar compras sueltas en vez de totales (la compra más grande no es de la categoría con más gasto) o escribir un `for` distinto para cada resultado.
</details>

<details><summary>🔒 Ejercicio 5: saldos paso a paso</summary>

```python
saldo = saldo_inicial_caja            # el estado arranca en el valor inicial
saldos_caja = []
for fecha, concepto, monto in movimientos_caja:
    saldo += monto                    # el monto 0 no cambia el saldo, pero el día cuenta
    saldos_caja.append(round(saldo, 2))
saldo_cierre = round(saldo, 2)
```

**Error típico:** arrancar en 0 y sumar el saldo inicial al final (el cierre sale bien, los intermedios no) o filtrar el monto 0 y perder un día de la lista.
</details>

<details><summary>🔒 Ejercicio 6: sobregiro, el primero, todos y las entradas</summary>

```python
# 1. Primer día en rojo: break
saldo = saldo_inicial_cuenta
primer_rojo = None
for fecha, concepto, monto in movimientos_cuenta:
    saldo += monto
    if saldo < 0:
        primer_rojo = fecha
        break                                   # ya tengo la respuesta

# 2 y 3. Todos los días en rojo y las entradas, en un solo recorrido
saldo = saldo_inicial_cuenta
saldo_anterior = saldo
dias_en_rojo = []
entradas_en_rojo = 0
for fecha, concepto, monto in movimientos_cuenta:
    saldo += monto                              # sin continue: el ajuste de 0 también se revisa
    if saldo < 0:
        dias_en_rojo.append(fecha)              # ESTAR en rojo
        if saldo_anterior >= 0:
            entradas_en_rojo += 1               # ENTRAR en rojo
    saldo_anterior = saldo                      # al final de la vuelta

# 4. Otra cuenta: si nunca pasa, queda None
saldo = saldo_inicial_ahorro
primer_rojo_ahorro = None
for fecha, concepto, monto in movimientos_ahorro:
    saldo += monto
    if saldo < 0:
        primer_rojo_ahorro = fecha
        break
```

**Error típico:** olvidar el `break` (te quedas con el último día), contar días en vez de entradas, o poner un `continue` en el monto 0 que se salta un día en rojo.
</details>

<details><summary>🔒 Ejercicio 7 🐛: el saldo mínimo</summary>

```python
# Parte A
saldo = saldo_inicial_agosto
saldo_minimo = None                              # centinela: aún no hay mínimo
fecha_saldo_minimo = None
for fecha, concepto, monto in movimientos_agosto:
    saldo += monto
    if saldo_minimo is None or saldo < saldo_minimo:   # `<`: en el empate gana el primero
        saldo_minimo = saldo
        fecha_saldo_minimo = fecha
saldo_minimo = round(saldo_minimo, 2)

# Parte B
pred_con_menor = "Proveedor B"         # con `<` el empate no reemplaza: queda el primero
pred_con_menor_igual = "Proveedor C"   # con `<=` el empate sí reemplaza: queda el último
```

**Error típico:** inicializar el mínimo en 0: si todos los saldos son positivos, nunca se actualiza. Con `<=` el empate lo gana el último día.
</details>

<details><summary>🔒 Ejercicio 8: kárdex del aceite</summary>

```python
stock = stock_inicial
fechas_bajo_minimo = []
fecha_recuperacion = None
mayor_salida = None
for movimiento in kardex:                        # un solo recorrido, cuatro resultados
    fecha, tipo, cantidad = movimiento
    stock_anterior = stock
    if tipo == "entrada":
        stock += cantidad
    else:
        stock -= cantidad
        if mayor_salida is None or cantidad > mayor_salida[2]:   # `>`: en el empate gana la primera
            mayor_salida = movimiento                            # registro completo
    if stock < stock_minimo:
        fechas_bajo_minimo.append(fecha)
    if fecha_recuperacion is None and stock_anterior < stock_minimo and stock > stock_minimo:
        fecha_recuperacion = fecha
stock_final = stock
```

**Error típico:** buscar la mayor cantidad entre todos los movimientos (incluye entradas), o detectar la recuperación sin el stock anterior.
</details>

<details><summary>🔒 Ejercicio 9: predicciones de comparación</summary>

```python
pred_a = False   # empatan en la fecha; decide 10 < 9
pred_b = False   # decide el índice 0: "b" < "a" es False; el 99 no se mira
pred_c = True    # empatan en 1; decide 500 < 1000 (son números)
pred_d = False   # sin ceros se compara "5" con "1" como caracteres
pred_e = True    # "1" < "2"
pred_f = True    # todo lo comparado empata: la más corta es menor
pred_min = ("2026-04-20", 40.0)   # la fecha más antigua; en el empate de fecha decide el monto
```

**Error típico:** leer textos como números (`"100" < "25"` es `True`) o mirar el monto cuando la fecha ya decidió.
</details>

<details><summary>🔒 Ejercicio 10 🐛: mayor ingreso, mayor egreso y top 3</summary>

```python
mayor_ingreso = max(movimientos_abril, key=lambda m: m[2])        # compara por el monto
mayor_egreso_abril = min(movimientos_abril, key=lambda m: m[2])   # el más negativo es el mínimo

egresos_abril = []
for movimiento in movimientos_abril:
    if movimiento[2] < 0:
        egresos_abril.append(movimiento)
top3_egresos = sorted(egresos_abril, key=lambda m: m[2])[:3]      # ascendente: el más negativo primero
```

**Error típico:** `min`/`max`/`sorted` sin `key` comparan por la fecha. Con montos negativos, `reverse=True` deja primero el egreso más pequeño.
</details>

<details><summary>🔒 Ejercicio 11: ranking de días con desempate</summary>

```python
venta_por_fecha = dict(zip(fechas_semana, ventas_dia))

registros = []
for numero, (fecha, venta, clientes) in enumerate(zip(fechas_semana, ventas_dia, clientes_dia), start=1):
    registros.append((numero, fecha, venta, clientes))
ranking = sorted(registros, key=lambda r: (-r[2], r[3]))   # venta desc; empate: menos clientes

dia_ganador = ranking[0][0]
fecha_ganadora = ranking[0][1]
```

**Error típico:** ordenar solo por venta: los empates quedan en el orden original y gana otro día. Sin `start=1` los números de día empiezan en 0.
</details>

<details><summary>🔒 Ejercicio 12: el mejor mes de cada local</summary>

```python
total_por_local = {}
mejor_mes_por_local = {}
for local, fila in zip(locales, ventas_mensuales):
    total_por_local[local] = sum(fila)
    mejor_j = 0
    for j in range(1, len(fila)):
        if fila[j] > fila[mejor_j]:          # `>`: en el empate se queda el primer mes
            mejor_j = j
    mejor_mes_por_local[local] = mejor_j + 1  # meses del 1 al 6

local_top = max(total_por_local, key=total_por_local.get)
```

**Error típico:** usar `>=` (en el empate gana el último mes) u olvidar sumar 1 a la posición.
</details>

<details><summary>🔒 Reto · Parte A: resuélvelo a tu manera</summary>

```python
# Saldo final: el estado arranca en saldo_inicial
saldo = saldo_inicial
for fecha, concepto, monto in movimientos:
    saldo += monto
saldo_final = round(saldo, 2)

# Conteos: el 0 no es ingreso ni egreso
n_ingresos = 0
n_egresos = 0
for fecha, concepto, monto in movimientos:
    if monto > 0:
        n_ingresos += 1
    elif monto < 0:
        n_egresos += 1

# Gasto por concepto: solo egresos, en positivo, redondeado al entregar
gasto_por_concepto = {}
for fecha, concepto, monto in movimientos:
    if monto < 0:
        gasto_por_concepto[concepto] = gasto_por_concepto.get(concepto, 0) - monto
for concepto in gasto_por_concepto:
    gasto_por_concepto[concepto] = round(gasto_por_concepto[concepto], 2)

# Mayor egreso: key compara por el monto; en el empate min devuelve el primero
mayor_egreso = min(movimientos, key=lambda m: m[2])

# Sobregiro: todos los días, las entradas y el primero
saldo = saldo_inicial
saldo_anterior = saldo
dias_sobregiro = []
veces_en_sobregiro = 0
primer_dia_sobregiro = None
for fecha, concepto, monto in movimientos:
    saldo += monto                          # sin continue: el ajuste de 0 también se revisa
    if saldo < 0:
        dias_sobregiro.append(fecha)
        if saldo_anterior >= 0:
            veces_en_sobregiro += 1
        if primer_dia_sobregiro is None:
            primer_dia_sobregiro = fecha
    saldo_anterior = saldo

# Concepto con más gasto
concepto_mas_gasto = max(gasto_por_concepto, key=gasto_por_concepto.get)
```

**Error típico:** empezar el saldo en 0, saltar el monto 0 con `continue` antes del chequeo, usar `<=` o `min(movimientos)` sin `key`, y contar días en sobregiro como si fueran entradas.
</details>

<details><summary>🔒 Reto · Parte B: un solo recorrido</summary>

```python
saldo = saldo_inicial
saldo_anterior = saldo
n_ingresos = 0
n_egresos = 0
gasto_por_concepto = {}
mayor_egreso = None
dias_sobregiro = []
veces_en_sobregiro = 0
primer_dia_sobregiro = None

for movimiento in movimientos:                    # UN solo recorrido, todos los acumuladores
    fecha, concepto, monto = movimiento
    saldo += monto

    if monto > 0:
        n_ingresos += 1
    elif monto < 0:
        n_egresos += 1
        gasto_por_concepto[concepto] = gasto_por_concepto.get(concepto, 0) - monto
        if mayor_egreso is None or monto < mayor_egreso[2]:     # `<`: en el empate gana el primero
            mayor_egreso = movimiento

    if saldo < 0:
        dias_sobregiro.append(fecha)
        if saldo_anterior >= 0:
            veces_en_sobregiro += 1
        if primer_dia_sobregiro is None:
            primer_dia_sobregiro = fecha
    saldo_anterior = saldo                         # al final de la vuelta

# Al entregar: lo que depende del diccionario ya completo
saldo_final = round(saldo, 2)
concepto_mas_gasto = None
for concepto, gasto in gasto_por_concepto.items():
    if concepto_mas_gasto is None or gasto > gasto_por_concepto[concepto_mas_gasto]:
        concepto_mas_gasto = concepto
for concepto in gasto_por_concepto:
    gasto_por_concepto[concepto] = round(gasto_por_concepto[concepto], 2)
```

**Error típico:** actualizar `saldo_anterior` antes de usarlo, o buscar el concepto con más gasto mientras el diccionario todavía se está llenando.
</details>

<details><summary>🔒 Nivel pro</summary>

```python
# 1. dict(zip(...)) en una línea, a partir de dos listas paralelas
fechas = []
saldos = []
saldo = saldo_inicial
for fecha, concepto, monto in movimientos:
    saldo += monto
    fechas.append(fecha)
    saldos.append(round(saldo, 2))
saldo_por_fecha = dict(zip(fechas, saldos))

# 2. Ranking: gasto descendente y, en el empate, nombre ascendente
gasto = {}
for fecha, concepto, monto in movimientos:
    if monto < 0:
        gasto[concepto] = gasto.get(concepto, 0) - monto
for concepto in gasto:
    gasto[concepto] = round(gasto[concepto], 2)
ranking_conceptos = sorted(gasto.items(), key=lambda kv: (-kv[1], kv[0]))

# 3. Racha: un contador que se reinicia y otro que guarda el mejor
racha = 0
racha_mas_larga = 0
for s in saldos:
    if s < 0:
        racha += 1
        if racha > racha_mas_larga:
            racha_mas_larga = racha
    else:
        racha = 0
```

**Error típico:** en la racha, contar todos los días en rojo sin reiniciar el contador al salir del rojo.
</details>